In [1]:
import os
import json
import re
import pandas as pd
import numpy as np
import networkx as nx
import chromadb
from pathlib import Path
from collections import Counter
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from llama_cpp import Llama, LlamaGrammar


RESULTS_DIR = Path('../results')
ATTCK_DIR   = Path('../data/attck')
CHROMA_DIR  = Path('../data/chroma')
MODEL_PATH  = '../models/qwen2.5-3b-instruct-q4_k_m.gguf'
RANDOM_SEED = 42


RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)



In [2]:
data_path = '../data/cicids'
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Monday (Benign only) and Wednesday(DoS only) are filtered outto work with a more balanced dataset
csv_files_filtered = [f for f in csv_files if ('Monday' not in f and 'Wednesday' not in f)]

print("Files read")
for f in csv_files_filtered:
    print(f"  - {f}")


Files read
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


In [3]:
# Combine all CSV files into one dataframe
dfs = []
for f in csv_files_filtered:
    file_path = os.path.join(data_path, f)
    df_temp = pd.read_csv(file_path)
    # Add source file column for reference
    df_temp['Source_File'] = f
    dfs.append(df_temp)
    print(f"Loaded {f}: {len(df_temp)} rows")

# Concatenate all dataframes
df_combined = pd.concat(dfs, ignore_index=True)

print(f"\nTotal combined rows: {len(df_combined)}")
print(f"Total columns: {len(df_combined.columns)}")

Loaded Tuesday-WorkingHours.pcap_ISCX.csv: 445909 rows
Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191033 rows
Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288602 rows
Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225745 rows
Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170366 rows
Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286467 rows

Total combined rows: 1608122
Total columns: 80


In [4]:
df_combined.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [5]:
print("\nLabel Counts:")
print(df_combined[' Label'].value_counts())


Label Counts:
 Label
BENIGN                        1303148
PortScan                       158930
DDoS                           128027
FTP-Patator                      7938
SSH-Patator                      5897
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Name: count, dtype: int64


In [6]:
# Clean the labels
df_combined[' Label'] = df_combined[' Label'].str.replace('�', '-', regex=False)
df_combined[' Label'] = df_combined[' Label'].str.replace(r'\s*-\s*', ' - ', regex=True)
df_combined[' Label'] = df_combined[' Label'].str.strip()

# Check the cleaned labels
print("Cleaned Label Counts:")
print(df_combined[' Label'].value_counts())

Cleaned Label Counts:
 Label
BENIGN                        1303148
PortScan                       158930
DDoS                           128027
FTP - Patator                    7938
SSH - Patator                    5897
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Name: count, dtype: int64


In [7]:
# Balanced sampling.
# Inclusion rule: a class must have at least CLASS_CAP rows to be kept.
# Classes below the cap are dropped as too sparse to contribute a full,
# balanced class to the evaluation. Surviving classes are downsampled to
# exactly CLASS_CAP so every class is equally represented.
#
# At CLASS_CAP = 1966 (the Bot row count), the surviving classes are:
#   BENIGN, PortScan, DDoS, FTP-Patator, SSH-Patator, Bot
# Dropped (below 1966): Web Attack - Brute Force (1507), Web Attack - XSS (652),
#   Infiltration (36), Web Attack - Sql Injection (21)
CLASS_CAP = 1966

if 'df_combined' not in globals():
    raise RuntimeError('`df_combined` not found. Run the data-loading cells first.')

sampled_dfs = []
kept_labels, dropped_labels = [], []

for label in df_combined[' Label'].unique():
    label_df = df_combined[df_combined[' Label'] == label]
    if len(label_df) < CLASS_CAP:
        dropped_labels.append((label, len(label_df)))
        continue
    sampled = label_df.sample(n=CLASS_CAP, random_state=42)
    sampled_dfs.append(sampled)
    kept_labels.append(label)

if len(sampled_dfs) == 0:
    df_sampled = pd.DataFrame(columns=df_combined.columns)
else:
    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

print(f"Class cap: {CLASS_CAP}")
print(f"Kept {len(kept_labels)} classes at {CLASS_CAP} rows each -> total sampled: {len(df_sampled)}")
print("\nDropped classes (below cap):")
for lbl, n in sorted(dropped_labels, key=lambda x: -x[1]):
    print(f"  {lbl:<30} {n}")
print("\nFinal label counts:")
print(df_sampled[' Label'].value_counts().to_string())


Class cap: 1966
Kept 6 classes at 1966 rows each -> total sampled: 11796

Dropped classes (below cap):
  Web Attack - Brute Force       1507
  Web Attack - XSS               652
  Infiltration                   36
  Web Attack - Sql Injection     21

Final label counts:
 Label
BENIGN           1966
FTP - Patator    1966
SSH - Patator    1966
Bot              1966
DDoS             1966
PortScan         1966


In [8]:
columns_to_keep = [
    ' Label',
    ' Destination Port',
    ' Flow Duration',
    ' Total Fwd Packets',
    ' Total Backward Packets',
    ' SYN Flag Count',
    ' RST Flag Count',
    'FIN Flag Count',
    'Flow Bytes/s',
    ' Flow Packets/s',
    ' Flow IAT Mean',
    ' Down/Up Ratio',
    ' Average Packet Size',
    ' Packet Length Std',
    'Active Mean',
    'Idle Mean'
]

df_filtered = df_sampled[columns_to_keep]
print(f"Filtered dataframe shape: {df_filtered.shape}")
print(f"Columns: {df_filtered.columns.tolist()}")

Filtered dataframe shape: (11796, 16)
Columns: [' Label', ' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', ' SYN Flag Count', ' RST Flag Count', 'FIN Flag Count', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Down/Up Ratio', ' Average Packet Size', ' Packet Length Std', 'Active Mean', 'Idle Mean']


In [9]:
# Load the ATT&CK STIX bundle once and expose `attck_techniques` dict for reuse
with open('../data/attck/enterprise-attack.json', 'r', encoding='utf-8') as f:
    bundle = json.load(f)

# Dictionary to store techniques keyed by MITRE ID
attck_techniques = {}

# Iterate through all objects in the bundle
for obj in bundle.get('objects', []):
    if obj.get('type') != 'attack-pattern':
        continue
    if obj.get('revoked') or obj.get('x_mitre_deprecated'):
        continue
    tid = None
    for ref in obj.get('external_references', []):
        if ref.get('source_name') == 'mitre-attack':
            tid = ref.get('external_id')
            break
    if not tid:
        continue
    name = obj.get('name', 'Unknown')
    tactics = []
    for phase in obj.get('kill_chain_phases', []):
        phase_name = phase.get('phase_name', '')
        if phase_name:
            tactics.append(phase_name.capitalize())
    attck_techniques[tid] = {
        'name': name,
        'tactics': tactics,
        'description': obj.get('description','')
    }

print("Sample techniques loaded from ATT&CK:\n")
sample_count = 0
for technique_id, technique_data in attck_techniques.items():
    if sample_count >= 3:
        break
    print(f"Technique ID: {technique_id}")
    print(f"  Name: {technique_data['name']}")
    print(f"  Tactics: {technique_data['tactics']}")
    print()
    sample_count += 1

Sample techniques loaded from ATT&CK:

Technique ID: T1055.011
  Name: Extra Window Memory Injection
  Tactics: ['Defense-evasion', 'Privilege-escalation']

Technique ID: T1053.005
  Name: Scheduled Task
  Tactics: ['Execution', 'Persistence', 'Privilege-escalation']

Technique ID: T1205.002
  Name: Socket Filters
  Tactics: ['Defense-evasion', 'Persistence', 'Command-and-control']



In [10]:
# Manual mapping from CICIDS labels to MITRE ATT&CK technique IDs
label_to_technique = {
    'BENIGN' : None,
    'FTP - Patator': 'T1110.001',      
    'SSH - Patator': 'T1110.001',       
    'DDoS': 'T1498.001',                
    'PortScan': 'T1046',                
    'Bot': 'T1071.001',                 
    'Web Attack - Brute Force': 'T1110.001', 
    'Web Attack - XSS': 'T1187',        
    'Infiltration': 'T1105',            
    'Web Attack - Sql Injection': 'T1190'  
}

In [11]:
# Map labels to MITRE technique IDs
df_sampled['Technique'] = df_sampled[' Label'].map(label_to_technique)

In [12]:
# retrieve tactics from technique ID using the single loaded ATT&CK dict
def get_tactics(tech_id):
    if pd.isna(tech_id):
        return None
    return attck_techniques.get(tech_id, {}).get('tactics', None)

df_sampled['Tactics'] = df_sampled['Technique'].apply(get_tactics)

In [13]:
#check if technique mapping is complete
# 2000 benign samples, hence remaining data is mapped
print(df_sampled['Technique'].isna().sum())

1966


In [14]:
df_sampled.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Source_File,Technique,Tactics
0,53,808,2,2,102,256,51,51,51.00,0.000000,...,0,0,0.0,0.0,0,0,BENIGN,Friday-WorkingHours-Morning.pcap_ISCX.csv,NaN,None
1,53,826,2,2,58,178,29,29,29.00,0.000000,...,0,0,0.0,0.0,0,0,BENIGN,Thursday-WorkingHours-Afternoon-Infilteration....,NaN,None
2,7815,38,1,1,0,0,0,0,0.00,0.000000,...,0,0,0.0,0.0,0,0,BENIGN,Tuesday-WorkingHours.pcap_ISCX.csv,NaN,None
3,443,5395624,16,23,772,27243,455,0,48.25,121.691687,...,0,0,0.0,0.0,0,0,BENIGN,Tuesday-WorkingHours.pcap_ISCX.csv,NaN,None
4,80,5882540,3,1,12,0,6,0,4.00,3.464102,...,0,0,0.0,0.0,0,0,BENIGN,Thursday-WorkingHours-Afternoon-Infilteration....,NaN,None


In [15]:
def get_port_service(port: int) -> str:
    """Maps common ports to service names for semantic grounding."""
    port_map = {
        20: "FTP-Data", 21: "FTP", 22: "SSH", 23: "Telnet", 25: "SMTP",
        53: "DNS", 67: "DHCP-Server", 68: "DHCP-Client", 69: "TFTP",
        80: "HTTP", 110: "POP3", 119: "NNTP", 123: "NTP", 135: "RPC",
        137: "NetBIOS-Name", 138: "NetBIOS-Datagram", 139: "NetBIOS-Session",
        143: "IMAP", 161: "SNMP", 162: "SNMP-Trap", 179: "BGP",
        389: "LDAP", 443: "HTTPS", 445: "SMB", 465: "SMTPS",
        514: "Syslog", 587: "SMTP-Submission", 636: "LDAPS",
        993: "IMAPS", 995: "POP3S", 1080: "SOCKS-Proxy", 1433: "MSSQL",
        1521: "OracleDB", 1723: "PPTP", 2049: "NFS", 3306: "MySQL",
        3389: "RDP", 5432: "PostgreSQL", 5900: "VNC", 6379: "Redis",
        8080: "HTTP-Alt", 8443: "HTTPS-Alt", 9000: "Dev-Services",
        9200: "Elasticsearch", 27017: "MongoDB"
    }
    return port_map.get(port, f"unknown-port-{port}")


PORT_TACTIC_HINTS = {
    21:   "possible credential access via brute-force (FTP)",
    22:   "possible credential access via brute-force (SSH)",
    80:   "possible web-based exploitation or injection (HTTP)",
    443:  "possible web-based exploitation or injection (HTTPS)",
    445:  "possible lateral movement via SMB file sharing",
    53:   "possible command-and-control via DNS tunneling",
    3389: "possible lateral movement via remote desktop (RDP)",
    8080: "possible web-based exploitation (HTTP alternate)",
}


def get_tactic_hint(port: int) -> str:
    return PORT_TACTIC_HINTS.get(port, "")


def describe_duration(dur: float) -> str:
    if dur < 1.0:   return "brief, sub-second connection"
    elif dur < 10.0:  return "short-lived connection"
    elif dur < 60.0:  return "sustained interaction"
    elif dur < 300.0: return "persistent session"
    else:             return "long-duration connection"


def describe_volume(fwd: int, bwd: int) -> str:
    total = fwd + bwd
    if total < 10:    return "minimal control exchange"
    elif total < 100:  return "light data transfer"
    elif total < 1000: return "moderate communication"
    else:              return "high-volume data stream"


def describe_flags(syn: int, rst: int, fin: int) -> str:
    if syn > 50 and rst == 0 and fin == 0:
        return "repeated connection attempts without completion (scanning or brute-force behavior)"
    elif rst > syn * 0.4:
        return "frequent connection resets (potential evasion, policy blocking, or connection instability)"
    elif fin > 0 and rst == 0:
        return "graceful connection termination"
    else:
        return "normal mixed TCP control behavior"


def describe_asymmetry(ratio: float) -> str:
    if ratio < 0.3:
        return "highly asymmetric outbound traffic (command flooding or data exfiltration pattern)"
    elif ratio > 3.0:
        return "highly asymmetric inbound traffic (bulk download or beacon response pattern)"
    else:
        return "balanced bidirectional communication"


def describe_iat(iat: float) -> str:
    if iat < 0.01:  return "extremely rapid, machine-speed pacing"
    elif iat < 0.1: return "fast automated timing"
    elif iat < 1.0: return "steady interactive pacing"
    else:           return "sporadic or human-like intervals"


def describe_session(act: float, idle: float) -> str:
    if act > 0 and idle < act * 0.1:
        return "continuous active session"
    elif idle > act * 2:
        return "intermittent beaconing with long dormant periods"
    else:
        return "regular active-idle communication cycle"


def describe_payload(avg: float, std: float) -> str:
    if avg < 100 and std < 50:
        return "small uniform control or command packets"
    elif avg < 100 and std >= 50:
        return "small but variable packets suggesting mixed command traffic"
    elif avg > 1000 and std < 200:
        return "large consistent data payload transfers"
    elif avg > 1000 and std >= 200:
        return "large variable-size packets suggesting mixed bulk and control data"
    else:
        return "moderate mixed-size packets"


def describe_rate(pps: float, bps: float) -> str:
    if pps > 1000 and bps > 1_000_000:
        return "extremely high packet and byte rate (flood or DoS pattern)"
    elif pps > 1000:
        return "high packet rate with low byte volume (SYN flood or scan pattern)"
    elif bps > 1_000_000:
        return "high throughput with moderate packet count (bulk transfer or exfiltration)"
    elif pps < 1 and bps < 100:
        return "very low rate suggesting slow-and-low reconnaissance or beaconing"
    else:
        return "moderate traffic rate"


In [16]:
def build_semantic_alert(row: pd.Series) -> str:
    port    = int(row[' Destination Port']) if pd.notna(row[' Destination Port']) else 0
    dur     = float(row[' Flow Duration']) if pd.notna(row[' Flow Duration']) else 0.0
    fwd     = int(row[' Total Fwd Packets']) if pd.notna(row[' Total Fwd Packets']) else 0
    bwd     = int(row[' Total Backward Packets']) if pd.notna(row[' Total Backward Packets']) else 0
    syn     = int(row[' SYN Flag Count']) if pd.notna(row[' SYN Flag Count']) else 0
    rst     = int(row[' RST Flag Count']) if pd.notna(row[' RST Flag Count']) else 0
    fin     = int(row['FIN Flag Count']) if pd.notna(row['FIN Flag Count']) else 0
    pps     = float(row[' Flow Packets/s']) if pd.notna(row[' Flow Packets/s']) else 0.0
    bps     = float(row['Flow Bytes/s']) if pd.notna(row['Flow Bytes/s']) else 0.0
    iat     = float(row[' Flow IAT Mean']) if pd.notna(row[' Flow IAT Mean']) else 0.0
    asym    = float(row[' Down/Up Ratio']) if pd.notna(row[' Down/Up Ratio']) else 0.0
    avg_sz  = float(row[' Average Packet Size']) if pd.notna(row[' Average Packet Size']) else 0.0
    std_sz  = float(row[' Packet Length Std']) if pd.notna(row[' Packet Length Std']) else 0.0
    act     = float(row['Active Mean']) if pd.notna(row['Active Mean']) else 0.0
    idle    = float(row['Idle Mean']) if pd.notna(row['Idle Mean']) else 0.0

    hint     = get_tactic_hint(port)
    hint_str = f" Threat context: {hint}." if hint else ""

    return (
        f"Network flow targeting {get_port_service(port)} on port {port}. "
        f"Connection profile: {describe_duration(dur)}. "
        f"Traffic exchange: {describe_volume(fwd, bwd)}, with {fwd} outbound and {bwd} inbound packets. "
        f"TCP behavior: {describe_flags(syn, rst, fin)}. "
        f"Timing pattern: {describe_iat(iat)}. "
        f"Traffic rate: {describe_rate(pps, bps)}. "
        f"Directionality: {describe_asymmetry(asym)}. "
        f"Session rhythm: {describe_session(act, idle)}. "
        f"Payload characteristics: {describe_payload(avg_sz, std_sz)}."
        f"{hint_str}"
    )


In [17]:
# Apply to the filtered DataFrame
df_filtered['alert_text'] = df_filtered.apply(build_semantic_alert, axis=1)
alert_texts = df_filtered['alert_text'].tolist()

In [18]:

pd.set_option('display.max_colwidth', None)
print(df_filtered['alert_text'].head())


0                                                                     Network flow targeting DNS on port 53. Connection profile: long-duration connection. Traffic exchange: minimal control exchange, with 2 outbound and 2 inbound packets. TCP behavior: normal mixed TCP control behavior. Timing pattern: sporadic or human-like intervals. Traffic rate: high packet rate with low byte volume (SYN flood or scan pattern). Directionality: balanced bidirectional communication. Session rhythm: regular active-idle communication cycle. Payload characteristics: moderate mixed-size packets. Threat context: possible command-and-control via DNS tunneling.
1                                                        Network flow targeting DNS on port 53. Connection profile: long-duration connection. Traffic exchange: minimal control exchange, with 2 outbound and 2 inbound packets. TCP behavior: normal mixed TCP control behavior. Timing pattern: sporadic or human-like intervals. Traffic rate: high packet rat

In [19]:
from sentence_transformers import SentenceTransformer, util
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(alert_texts, convert_to_tensor=True)
communities = util.community_detection(embeddings, min_community_size=2, threshold=0.75)
print(f"Number of communities: {len(communities)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of communities: 21


In [20]:
# assign community labels back to the DataFrame
# build positional array and assign
comm_ids = np.full(len(df_filtered), -1, dtype=int)
for cid, members in enumerate(communities):
    comm_ids[np.asarray(members, dtype=int)] = cid
df_filtered['community_id'] = comm_ids

# validations
assigned_count = (df_filtered['community_id'] != -1).sum()
assert assigned_count == sum(len(c) for c in communities), "Assigned count mismatch"
flat = [i for c in communities for i in c]
assert len(set(flat)) == len(flat), "Overlapping indices in communities"
print("Assigned", assigned_count, "clustered rows; unclustered =", (len(df_filtered)-assigned_count))
print(df_filtered['community_id'].value_counts(dropna=False).head(10))

Assigned 11793 clustered rows; unclustered = 3
community_id
0    6548
1    2630
2     929
3     772
4     405
5     234
6      76
7      67
8      66
9      13
Name: count, dtype: int64


In [21]:
# Evaluation: Label Purity and Intra-Cluster Similarity
import numpy as np
import torch
from sentence_transformers import util as sutil

if 'embeddings' not in globals():
    raise RuntimeError('embeddings tensor not found — re-run the SentenceTransformer encoding cell first.')

def compute_label_purity(df, community_col='community_id', label_col=' Label'):
    """For each community, dominant label count / community size. Mean across communities."""
    purities = []
    for cid, group in df[df[community_col] != -1].groupby(community_col):
        dominant = group[label_col].value_counts().iloc[0]
        purities.append(dominant / len(group))
    return round(float(np.mean(purities)), 4)

def compute_intra_cluster_similarity(df, embeddings_tensor, community_col='community_id', max_sample=100):
    """Mean pairwise cosine similarity within each community.
    Sampled to max_sample per community for speed. Vectorised via pytorch_cos_sim."""
    sims = []
    for cid, group in df[df[community_col] != -1].groupby(community_col):
        indices = group.index.tolist()
        indices = [i for i in indices if i < len(embeddings_tensor)]
        if len(indices) < 2:
            continue
        if len(indices) > max_sample:
            indices = indices[:max_sample]
        comm_embs = embeddings_tensor[indices]
        sim_matrix = sutil.pytorch_cos_sim(comm_embs, comm_embs)
        # upper triangle mask — excludes diagonal (self-similarity = 1.0)
        mask = torch.triu(torch.ones(sim_matrix.shape, dtype=torch.bool), diagonal=1)
        sims.append(float(sim_matrix[mask].mean().item()))
    return round(float(np.mean(sims)), 4)

label_purity_score = compute_label_purity(df_filtered)
print('Label purity done.')

intra_sim_score = compute_intra_cluster_similarity(df_filtered, embeddings)
print('Intra-cluster similarity done.')

clustering_metrics = {
    'n_communities':            int(df_filtered[df_filtered['community_id'] != -1]['community_id'].nunique()),
    'n_unclustered':            int((df_filtered['community_id'] == -1).sum()),
    'label_purity':             label_purity_score,
    'intra_cluster_similarity': intra_sim_score,
}

with open(RESULTS_DIR / 'clustering_metrics.json', 'w') as f:
    json.dump(clustering_metrics, f, indent=2)

print('\nCLUSTERING METRICS')
print(f"  Communities detected:      {clustering_metrics['n_communities']}")
print(f"  Unclustered alerts:        {clustering_metrics['n_unclustered']}")
print(f"  Label Purity:              {label_purity_score:.4f}  (1.0 = perfectly pure clusters)")
print(f"  Intra-Cluster Similarity:  {intra_sim_score:.4f}  (1.0 = all members identical)")

Label purity done.
Intra-cluster similarity done.

CLUSTERING METRICS
  Communities detected:      21
  Unclustered alerts:        3
  Label Purity:              0.8131  (1.0 = perfectly pure clusters)
  Intra-Cluster Similarity:  0.9440  (1.0 = all members identical)


In [22]:
df_filtered['Technique'] = df_sampled.loc[df_filtered.index, 'Technique'].values
df_filtered['Tactics']   = df_sampled.loc[df_filtered.index, 'Tactics'].apply(str).values
# save the processed DataFrame with community labels and technique mappings for future analysis
RESULTS_DIR = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(RESULTS_DIR / 'cicids_processed.csv', index=False)
print(f'Saved {len(df_filtered)} rows to cicids_processed.csv')

Saved 11796 rows to cicids_processed.csv


In [23]:
# LLM initialization (uses Llama imported at top)
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096, n_gpu_layers=-1, n_threads=8,
    n_batch=256, verbose=False, seed=RANDOM_SEED,
    )
print(f'LLM loaded: {MODEL_PATH}')

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded: ../models/qwen2.5-3b-instruct-q4_k_m.gguf


In [24]:
SECURITY_RELATIONS = [
    'PERFORMS_RECONNAISSANCE',
    'PERFORMS_PORT_SCAN',
    'BRUTE_FORCES_CREDENTIAL',
    'ACCESS_CREDENTIALS',
    'EXPLOITS_VULNERABILITY',
    'ESTABLISHES_C2',
    'PERFORMS_BEACONING',
    'CAUSES_DENIAL_OF_SERVICE',
    'MOVES_LATERALLY',
    'EXFILTRATES_DATA',
    'EXECUTES_PAYLOAD',
]

TRIPLE_SCHEMA = {
    'type': 'object',
    'properties': {
        'triples': {
            'type': 'array', 'minItems': 1, 'maxItems': 4,
            'items': {
                'type': 'object',
                'properties': {
                    'subject':  {'type': 'string'},
                    'relation': {'type': 'string', 'enum': SECURITY_RELATIONS},
                    'target':   {'type': 'string'}
                },
                'required': ['subject', 'relation', 'target']
            }
        }
    },
    'required': ['triples']
}

grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print(f'Grammar ready — {len(SECURITY_RELATIONS)} constrained relations')

Grammar ready — 11 constrained relations


In [25]:
RELATION_GUIDE = """
Relation definitions (use exactly as written):
  PERFORMS_RECONNAISSANCE  : active discovery, general service enumeration and scanning
  PERFORMS_PORT_SCAN       : focused port/protocol probing across multiple ports or hosts
  BRUTE_FORCES_CREDENTIAL  : repeated authentication attempts against a service (password guessing)
  ACCESS_CREDENTIALS       : theft or collection of credentials (beyond brute-force, e.g., credential dumping or theft)
  EXPLOITS_VULNERABILITY   : exploitation of a software or configuration flaw
  ESTABLISHES_C2           : outbound communication to a command-and-control channel
  PERFORMS_BEACONING       : periodic callback/beacon patterns indicative of persistent C2
  CAUSES_DENIAL_OF_SERVICE : flooding or resource exhaustion of a target service
  MOVES_LATERALLY          : accessing internal hosts after initial compromise
  EXFILTRATES_DATA         : transferring data out of the environment
  EXECUTES_PAYLOAD         : running code or commands on a target
""".strip()


def normalise_entity(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9_]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text[:80]


In [26]:
# Dynamic example retrieval was tested here and reverted: at 3B scale the
# extractor copies demonstration structure rather than weighing flow evidence,
# which degraded the reliable SSH brute-force extractions. Fixed anchored
# examples in extract_triples performed better. Kept as a documented negative result.


In [27]:
def extract_triples(alert_texts: list, community_id: int) -> list:
    """Extracts validated triple(s) from a list of alert_texts.
    Uses the LLM grammar to constrain relation values. Returns a list of dictionaries
    with keys: subject, relation, target.
    """
    if 'llm' not in globals():
        raise RuntimeError('LLM not initialized. Run the cell that creates `llm` before extracting triples.')
    block = '\n'.join(f'- {t}' for t in alert_texts[:6])

    assert ' Label' not in block and 'Technique' not in block, f"Potential label leakage in community {community_id}"

    prompt = f"""[INST] You are a cybersecurity analyst performing threat analysis.
Read the network alerts below and extract between 1 and 4 semantic triples describing
the attack behaviour using standard security terminology.

{RELATION_GUIDE}

For each triple:
- subject : name the specific actor observed (e.g. 'ssh_scanner', 'ddos_flood_source')
- relation: choose the ONE relation that best fits — use payload size, flag patterns,
            packet rate, IAT timing, and session rhythm to decide
- target  : name the specific service or resource (e.g. 'ssh_port_22', 'http_web_server')

Name what you observe. No generic placeholders like 'source' or 'destination'.

Example 1 (SSH brute-force):
{{"triples": [
  {{"subject": "ssh_brute_force_client",      "relation": "BRUTE_FORCES_CREDENTIAL", "target": "ssh_port_22_service"}},
  {{"subject": "ssh_brute_force_client",      "relation": "PERFORMS_RECONNAISSANCE", "target": "ssh_authentication_endpoint"}},
  {{"subject": "credential_guessing_process", "relation": "EXECUTES_PAYLOAD",        "target": "password_spray_module"}},
  {{"subject": "attacker",                    "relation": "BRUTE_FORCES_CREDENTIAL", "target": "user_account_store"}}
]}}

Example 2 (DDoS flood):
{{"triples": [
  {{"subject": "flood_source",  "relation": "CAUSES_DENIAL_OF_SERVICE", "target": "http_web_server"}},
  {{"subject": "flood_source",  "relation": "PERFORMS_RECONNAISSANCE",  "target": "target_host_availability"}},
  {{"subject": "botnet_node",   "relation": "ESTABLISHES_C2",           "target": "c2_callback_channel"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY valid JSON. [/INST]"""
    out = llm(prompt, max_tokens=512, temperature=0, seed=RANDOM_SEED,
              grammar=grammar, repeat_penalty=1.1, stop=['[/INST]'])
    raw = out['choices'][0]['text'].strip()

    try:
        triples = json.loads(raw).get('triples', [])
    except Exception as e:
        print(f'  [Community {community_id}] parse failed: {e}')
        triples = []

    valid_relations = set(SECURITY_RELATIONS)
    validated = []
    for t in triples:
        subj = normalise_entity(str(t.get('subject', '')))
        rel  = str(t.get('relation', '')).upper().strip()
        tgt  = normalise_entity(str(t.get('target', '')))
        if not subj or not tgt:
            continue
        if rel not in valid_relations:
            rel = 'PERFORMS_RECONNAISSANCE'
        validated.append({'subject': subj, 'relation': rel, 'target': tgt})

    return validated


In [28]:
MIN_COMMUNITY_SIZE = 3

community_df = df_filtered[df_filtered['community_id'] != -1].copy()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
community_df.to_csv(RESULTS_DIR / 'community_assignments.csv', index=False)
print(f'Prepared community_df with {len(community_df)} clustered alerts across {community_df["community_id"].nunique()} communities')

print('Warming up LLM...')
_ = llm('[INST] Test. [/INST]', max_tokens=5, temperature=0, seed=RANDOM_SEED)
print('Starting extraction\n')

community_triples = {}

for cid, group in community_df.groupby('community_id'):
    if len(group) < MIN_COMMUNITY_SIZE:
        community_triples[str(int(cid))] = []
        print(f'Community {cid} skipped — size {len(group)} < {MIN_COMMUNITY_SIZE}')
        continue
    texts   = group['alert_text'].dropna().tolist()
    triples = extract_triples(texts, int(cid))
    community_triples[str(int(cid))] = triples

    dom_label = group[' Label'].mode().iloc[0]
    print(f'Community {cid} [{dom_label}]:')
    for t in triples:
        print(f'  {t["subject"]} --[{t["relation"]}]-- {t["target"]}')
    print()

print(f'Extraction complete: {len(community_triples)} communities')


Prepared community_df with 11793 clustered alerts across 21 communities
Warming up LLM...
Starting extraction

Community 0 [FTP - Patator]:
  unknown_port_7815_scanner --[PERFORMS_PORT_SCAN]-- unknown_port_7815_service
  unknown_port_35143_scanner --[PERFORMS_PORT_SCAN]-- unknown_port_35143_service
  http_web_server_exploiter --[EXPLOITS_VULNERABILITY]-- http_web_server
  unknown_port_59408_scanner --[PERFORMS_PORT_SCAN]-- unknown_port_59408_service

Community 1 [DDoS]:
  http_exploiter --[EXPLOITS_VULNERABILITY]-- http_web_server
  http_exploiter --[PERFORMS_RECONNAISSANCE]-- web_service_availability
  http_exploiter --[CAUSES_DENIAL_OF_SERVICE]-- http_web_server
  http_exploiter --[ESTABLISHES_C2]-- c2_channel

Community 2 [SSH - Patator]:
  ssh_brute_force_client --[BRUTE_FORCES_CREDENTIAL]-- ssh_port_22_service
  ssh_brute_force_client --[PERFORMS_RECONNAISSANCE]-- ssh_authentication_endpoint
  credential_guessing_process --[EXECUTES_PAYLOAD]-- password_spray_module
  attacker --[B

In [29]:
def generate_baseline_report(raw_triples: str, alert_context: str=None, cid: int=None) -> dict:
    """
    True no-retrieval baseline: LLM generates a report from alert text and triples alone,
    with zero ChromaDB retrieval.
    """
    import re, json as _json
    if not raw_triples:
        print(f'  [Baseline Community {cid}] skipped — no triples')
        return {'technique_id': 'Unknown', 'tactic': '', 'summary': '', 'evidence': ''}

    alert_context = (alert_context or '')[:400]
    if isinstance(raw_triples, list):
        triples_short = '\n'.join(
            f"{t.get('subject','')} --[{t.get('relation','')}]--> {t.get('target','')}"
            for t in raw_triples
        )[:400]
    else:
        triples_short = str(raw_triples)[:400]

    prompt = f"""[INST] You are a cybersecurity analyst. Respond with valid JSON only. No markdown, no explanation, no extra text.
Based on the network behaviors and relationships below, identify the MITRE ATT&CK technique.

ALERT CONTEXT:
{alert_context}

RELATIONSHIPS:
{triples_short}

Reply with exactly this JSON and nothing else:
{{"technique_id": "T1046", "tactic": "Discovery", "summary": "brief description here", "evidence": "supporting evidence here"}} [/INST]
{{"""

    try:
        out = llm(prompt, max_tokens=256, temperature=0.0,
                  seed=RANDOM_SEED, repeat_penalty=1.0,
                  stop=['<|im_end|>', '<|im_start|>'])
        raw = '{' + out['choices'][0]['text']
        # cut off anything after the first complete JSON object
        match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
        raw = match.group(0) if match else raw
        print(f'  [Baseline raw]: {repr(raw[:200])}')
        parsed = _json.loads(raw)
        return {
            'technique_id': str(parsed.get('technique_id', 'Unknown')),
            'tactic':       str(parsed.get('tactic', '')),
            'summary':      str(parsed.get('summary', ''))[:400],
            'evidence':     str(parsed.get('evidence', '')),
        }
    except Exception as e:
        print(f'  [Baseline Community {cid}] parse failed: {e} | raw: {repr(raw[:150]) if "raw" in dir() else "no output"}')
        return {'technique_id': 'Unknown', 'tactic': '', 'summary': '', 'evidence': ''}

In [30]:
all_subjects = set()
all_targets  = set()
rel_counts   = Counter()
valid_count  = total_count = 0

for triples in community_triples.values():
    for t in triples:
        total_count += 1
        if all(k in t and t[k] for k in ['subject', 'relation', 'target']):
            valid_count += 1
            all_subjects.add(t['subject'])
            all_targets.add(t['target'])
            rel_counts[t['relation']] += 1

triple_metrics = {
    'n_communities':         len(community_triples),
    'total_triples':         total_count,
    'valid_triples':         valid_count,
    'valid_ratio':           round(valid_count / max(total_count, 1), 4),
    'unique_subjects':       len(all_subjects),
    'unique_targets':        len(all_targets),
    'unique_entities_total': len(all_subjects | all_targets),
    'relation_counts':       dict(rel_counts),
    'relation_coverage':     f'{len(rel_counts)}/{len(SECURITY_RELATIONS)} ontology relations used',
}
with open(RESULTS_DIR / 'community_triples.json', 'w', encoding='utf-8') as f:
    json.dump(community_triples, f, indent=2)
with open(RESULTS_DIR / 'triple_metrics.json', 'w') as f:
    json.dump(triple_metrics, f, indent=2)

print('TRIPLE METRICS')
for k, v in triple_metrics.items():
    print(f'  {k}: {v}')

TRIPLE METRICS
  n_communities: 21
  total_triples: 80
  valid_triples: 80
  valid_ratio: 1.0
  unique_subjects: 32
  unique_targets: 46
  unique_entities_total: 74
  relation_counts: {'PERFORMS_PORT_SCAN': 13, 'EXPLOITS_VULNERABILITY': 3, 'PERFORMS_RECONNAISSANCE': 16, 'CAUSES_DENIAL_OF_SERVICE': 11, 'ESTABLISHES_C2': 13, 'BRUTE_FORCES_CREDENTIAL': 7, 'EXECUTES_PAYLOAD': 7, 'PERFORMS_BEACONING': 2, 'EXFILTRATES_DATA': 3, 'MOVES_LATERALLY': 5}
  relation_coverage: 10/11 ontology relations used


In [31]:
# Build graph reusing previously-loaded ATT&CK techniques and add bridge edges 
G = nx.DiGraph()

# Behavioral layer
edge_weights     = {}
edge_communities = {}
total_triples    = 0
invalid_triples  = 0

for cid, triples in community_triples.items():
    for t in triples:
        s = str(t.get('subject',  '')).strip()
        r = str(t.get('relation', '')).strip()
        o = str(t.get('target',   '')).strip()
        if not s or not r or not o:
            invalid_triples += 1
            continue
        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), weight in edge_weights.items():
    G.add_node(src, layer='behavioral')
    G.add_node(tgt, layer='behavioral')
    G.add_edge(src, tgt, relation=rel, weight=weight, layer='behavioral',
               communities=','.join(edge_communities[(src, rel, tgt)]))

# ATT&CK layer — rely on the single `attck_techniques` loaded earlier
if 'attck_techniques' not in globals():
    raise RuntimeError('ATT&CK techniques not loaded — run the ATT&CK load cell first.')
else:
    print(f'Reusing existing `attck_techniques` ({len(attck_techniques)})')

# Add ATT&CK nodes idempotently
for tid, info in attck_techniques.items():
    if tid not in G:
        G.add_node(tid, layer='attck', name=info.get('name',''),
                   tactic=(info.get('tactics') or [''])[0], description=info.get('description',''))


RELATION_TO_TECHNIQUES = {
    'PERFORMS_RECONNAISSANCE':  ['T1046',    'T1595',    'T1590'],
    'PERFORMS_PORT_SCAN':       ['T1046',    'T1595'],
    'BRUTE_FORCES_CREDENTIAL':  ['T1110',    'T1110.001','T1110.003'],
    'ACCESS_CREDENTIALS':       ['T1555',    'T1078',    'T1110'],
    'EXPLOITS_VULNERABILITY':   ['T1190',    'T1203'],
    'ESTABLISHES_C2':           ['T1071',    'T1071.001','T1071.004'],
    'PERFORMS_BEACONING':       ['T1071',    'T1071.004'],
    'CAUSES_DENIAL_OF_SERVICE': ['T1498',    'T1499',    'T1499.001'],
    'MOVES_LATERALLY':          ['T1021',    'T1570'],
    'EXFILTRATES_DATA':         ['T1041',    'T1048'],
    'EXECUTES_PAYLOAD':         ['T1059',    'T1059.007','T1105'],
}
print('Inserted default RELATION_TO_TECHNIQUES mapping')

# connect behavioural nodes to ATT&CK technique nodes using RELATION_TO_TECHNIQUES mapping
bridge_count = 0
for (src, rel, tgt), weight in edge_weights.items():
    for tid in RELATION_TO_TECHNIQUES.get(rel, []):
        if tid in G and not G.has_edge(src, tid):
            G.add_edge(src, tid, relation='ASSOCIATED_WITH', weight=weight,
                       layer='bridge', communities=','.join(edge_communities.get((src, rel, tgt), [])))
            bridge_count += 1

b_nodes = sum(1 for n, d in G.nodes(data=True) if d.get('layer') == 'behavioral')
b_edges = sum(1 for _, _, d in G.edges(data=True) if d.get('layer') == 'behavioral')
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'  Behavioral : {b_nodes} nodes, {b_edges} edges')
print(f'  ATT&CK     : {len(attck_techniques)} technique nodes')
print(f'  Bridge     : {bridge_count} edges | Skipped: {invalid_triples} invalid triples')

Reusing existing `attck_techniques` (691)
Inserted default RELATION_TO_TECHNIQUES mapping
Graph: 765 nodes, 234 edges
  Behavioral : 74 nodes, 70 edges
  ATT&CK     : 691 technique nodes
  Bridge     : 164 edges | Skipped: 0 invalid triples


In [32]:
beh_nodes = [(n, d) for n, d in G.degree() if G.nodes[n].get('layer') == 'behavioral']
print('TOP 10 BEHAVIORAL NODES BY DEGREE')
for node, deg in sorted(beh_nodes, key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {node} (degree={deg})')

print('\nTOP 10 BEHAVIORAL EDGES BY WEIGHT')
beh_edges = [(u, v, d['relation'], d['weight']) for u, v, d in G.edges(data=True)
             if d.get('layer') == 'behavioral']
for u, v, rel, w in sorted(beh_edges, key=lambda x: x[3], reverse=True)[:10]:
    print(f'  (w={w}) {u} --[{rel}]--> {v}')

reachable_attck = [n for n in G.nodes()
                   if G.nodes[n].get('layer') == 'attck' and G.in_degree(n) > 0]
print(f'\nATT&CK nodes reachable: {len(reachable_attck)}')
for tid in sorted(reachable_attck):
    print(f"  {tid} | {G.nodes[tid].get('name')} | {G.nodes[tid].get('tactic')}")

TOP 10 BEHAVIORAL NODES BY DEGREE
  dns_tunneling_actor (degree=18)
  attacker (degree=15)
  smb_spoofing_client (degree=15)
  ldap_flood_source (degree=15)
  http_exploiter (degree=14)
  smtp_suspect (degree=14)
  smb_spoofing_actor (degree=14)
  mysql_scanner (degree=12)
  imap_scanner (degree=12)
  https_exploiter (degree=11)

TOP 10 BEHAVIORAL EDGES BY WEIGHT
  (w=3) credential_guessing_process --[EXECUTES_PAYLOAD]--> password_spray_module
  (w=2) ssh_brute_force_client --[PERFORMS_RECONNAISSANCE]--> ssh_authentication_endpoint
  (w=2) attacker --[BRUTE_FORCES_CREDENTIAL]--> user_account_store
  (w=1) unknown_port_7815_scanner --[PERFORMS_PORT_SCAN]--> unknown_port_7815_service
  (w=1) unknown_port_35143_scanner --[PERFORMS_PORT_SCAN]--> unknown_port_35143_service
  (w=1) http_web_server_exploiter --[EXPLOITS_VULNERABILITY]--> http_web_server
  (w=1) unknown_port_59408_scanner --[PERFORMS_PORT_SCAN]--> unknown_port_59408_service
  (w=1) http_exploiter --[CAUSES_DENIAL_OF_SERVICE]--

In [33]:
nx.write_graphml(G, RESULTS_DIR / 'knowledge_graph.graphml')

nodes_out = [{'node': n, 'degree': G.degree(n), 'layer': d.get('layer', ''),
              'name': d.get('name', ''), 'tactic': d.get('tactic', ''),
              'description': d.get('description', '')}
             for n, d in G.nodes(data=True)]
pd.DataFrame(nodes_out).sort_values('degree', ascending=False).to_csv(
    RESULTS_DIR / 'knowledge_graph_nodes.csv', index=False)

edges_out = [{'source': u, 'target': v, 'relation': d['relation'],
              'weight': d['weight'], 'layer': d.get('layer', ''),
              'communities': d.get('communities', '')}
             for u, v, d in G.edges(data=True)]
kg_edges_df = pd.DataFrame(edges_out).sort_values('weight', ascending=False)
kg_edges_df.to_csv(RESULTS_DIR / 'knowledge_graph_edges.csv', index=False)

rel_dist = Counter(d['relation'] for _, _, d in G.edges(data=True))
kg_metrics = {
    'total_triples_processed': total_triples,
    'invalid_triples_skipped': invalid_triples,
    'behavioral_nodes':        b_nodes,
    'attck_technique_nodes':   len(attck_techniques),
    'total_nodes':             G.number_of_nodes(),
    'behavioral_edges':        b_edges,
    'bridge_edges':            bridge_count,
    'total_edges':             G.number_of_edges(),
    'attck_nodes_reachable':   len(reachable_attck),
    'relation_distribution':   dict(rel_dist),
}
with open(RESULTS_DIR / 'knowledge_graph_metrics.json', 'w') as f:
    json.dump(kg_metrics, f, indent=2)

print('Graph saved. Metrics:')
for k, v in kg_metrics.items():
    print(f'  {k}: {v}')

Graph saved. Metrics:
  total_triples_processed: 80
  invalid_triples_skipped: 0
  behavioral_nodes: 74
  attck_technique_nodes: 691
  total_nodes: 765
  behavioral_edges: 70
  bridge_edges: 164
  total_edges: 234
  attck_nodes_reachable: 21
  relation_distribution: {'PERFORMS_PORT_SCAN': 13, 'ASSOCIATED_WITH': 164, 'EXPLOITS_VULNERABILITY': 2, 'CAUSES_DENIAL_OF_SERVICE': 11, 'PERFORMS_RECONNAISSANCE': 12, 'ESTABLISHES_C2': 13, 'BRUTE_FORCES_CREDENTIAL': 4, 'EXECUTES_PAYLOAD': 5, 'PERFORMS_BEACONING': 2, 'MOVES_LATERALLY': 5, 'EXFILTRATES_DATA': 3}


In [34]:
EMBED_MODEL       = 'all-MiniLM-L6-v2'
TOP_K             = 10
MAX_CHARS_PER_DOC = 400

# Maps each SECURITY_RELATION to the ATT&CK tactic scope it belongs to.
# Used at query time to restrict ChromaDB retrieval to the relevant tactic,
# preventing the model from retrieving semantically similar but tactically wrong techniques.
RELATION_TO_TACTIC_SCOPE = {
    'PERFORMS_RECONNAISSANCE':  'discovery',
    'PERFORMS_PORT_SCAN':       'discovery',
    'BRUTE_FORCES_CREDENTIAL':  'credential-access',
    'ACCESS_CREDENTIALS':       'credential-access',
    'EXPLOITS_VULNERABILITY':   'initial-access',
    'ESTABLISHES_C2':           'command-and-control',
    'PERFORMS_BEACONING':       'command-and-control',
    'CAUSES_DENIAL_OF_SERVICE': 'impact',
    'MOVES_LATERALLY':          'lateral-movement',
    'EXFILTRATES_DATA':         'exfiltration',
    'EXECUTES_PAYLOAD':         'execution',
}

ef         = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))

client.delete_collection('attack_techniques')
collection = client.get_or_create_collection(name='attack_techniques', embedding_function=ef)

docs, ids, metas = [], [], []
for tid, info in attck_techniques.items():
    tactic_list = info.get('tactics') or ['']
    tactic      = tactic_list[0] if isinstance(tactic_list, list) else (tactic_list or '')
    tactic_norm = tactic.lower().replace(' ', '-')
    text = (f"ID: {tid}\nName: {info.get('name','')}\n"
            f"Tactic: {tactic}\nDescription: {info.get('description','')}")
    docs.append(text)
    ids.append(tid)
    metas.append({'technique_id': tid, 'name': info.get('name',''), 'tactic': tactic_norm})

collection.add(ids=ids, documents=docs, metadatas=metas)
print(f'ChromaDB ready: {collection.count()} techniques')

# Build a tactic -> ordered technique-id list so the local rerank can be
# restricted to the same tactic scope used for ChromaDB retrieval.
tactic_to_tids = {}
for tid, info in attck_techniques.items():
    tactic_list = info.get('tactics') or ['']
    for tac in (tactic_list if isinstance(tactic_list, list) else [tactic_list]):
        tac_norm = (tac or '').lower().replace(' ', '-')
        tactic_to_tids.setdefault(tac_norm, []).append(tid)
print(f'Tactic scopes indexed: {len(tactic_to_tids)}')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB ready: 691 techniques
Tactic scopes indexed: 14


In [35]:
REPORT_SCHEMA = {
    'type': 'object',
    'properties': {
        'technique_id': {'type': 'string'},
        'tactic':       {'type': 'string'},
        'summary':      {'type': 'string'},
        'evidence':     {'type': 'string'},
        'next_step':    {'type': 'string'}
    },
    'required': ['technique_id', 'tactic', 'summary', 'evidence', 'next_step']
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))
print('Report grammar ready.')

Report grammar ready.


In [36]:
def infer_tactic_scope(triples: list) -> str | None:
    """Returns the most frequent tactic scope implied by the community's relations,
    or None if no dominant scope can be determined.
    Used to pre-filter ChromaDB to the relevant tactic before semantic search.
    """
    scope_counts = Counter()
    for t in triples:
        rel   = t.get('relation', '')
        scope = RELATION_TO_TACTIC_SCOPE.get(rel)
        if scope:
            scope_counts[scope] += 1
    return scope_counts.most_common(1)[0][0] if scope_counts else None


def build_hyde_query(cid: int):
    """Returns (hyde_text, raw_triple_text, alert_context, tactic_scope). No labels passed anywhere."""
    group   = community_df[community_df['community_id'] == cid]
    triples = community_triples.get(str(cid), [])

    triple_sentences = []
    for t in triples:
        s = t.get('subject',  '').replace('_', ' ')
        r = t.get('relation', '').replace('_', ' ').lower()
        o = t.get('target',   '').replace('_', ' ')
        if s and r and o:
            triple_sentences.append(f'{s} {r} {o}')
    raw_triple_text = '. '.join(triple_sentences)

    community_entities = ({t.get('subject', '') for t in triples} |
                          {t.get('target',  '') for t in triples})
    related = kg_edges_df[
        (kg_edges_df['source'].isin(community_entities) |
         kg_edges_df['target'].isin(community_entities)) &
        (kg_edges_df['layer'] == 'behavioral')
    ].head(3)
    graph_ctx = '; '.join(
        f"{r.source.replace('_',' ')} {r.relation.replace('_',' ').lower()} {r.target.replace('_',' ')}"
        for _, r in related.iterrows()
    ) or 'None identified'

    tactic_scope = infer_tactic_scope(triples)

    hyde_prompt = f"""[INST] You are a cybersecurity threat analyst.
Write a concise technical description of the attack technique based on the observed
network behaviours below. Write in MITRE ATT&CK style: what the adversary does,
which resources they target, observable indicators. Do NOT name a specific technique ID.

OBSERVED BEHAVIOURS:
{raw_triple_text}

RECURRING GRAPH PATTERNS:
{graph_ctx}

Write 3 to 5 sentences. [/INST]"""

    out = llm(hyde_prompt, max_tokens=200, temperature=0.2,
              seed=RANDOM_SEED, repeat_penalty=1.1, stop=['[/INST]'])
    hyde_query   = out['choices'][0]['text'].strip()
    alert_context = ' '.join(group['alert_text'].dropna().head(3).tolist())

    assert ' Label' not in raw_triple_text and 'Technique' not in raw_triple_text, \
        f"Potential label leakage detected in raw_triple_text for community {cid}"
    assert ' Label' not in alert_context and 'Technique' not in alert_context, \
        f"Potential label leakage detected in alert_context for community {cid}"

    return hyde_query, raw_triple_text, alert_context, tactic_scope


In [37]:
def generate_rag_report(hyde_query, raw_triples, alert_context, docs, metas, candidate_id=None):
    top_id = candidate_id or 'Unknown'
    tactic = ''
    try:
        if candidate_id and candidate_id in attck_techniques:
            tl = attck_techniques[candidate_id].get('tactics') or ['']
            tactic = (tl[0] if isinstance(tl, list) else tl).lower().replace(' ', '-')
        elif metas and isinstance(metas, list):
            found = next((m for m in metas if isinstance(m, dict) and m.get('technique_id')), None)
            if found:
                top_id = found.get('technique_id', 'Unknown')
                tactic = found.get('tactic', '')
    except Exception:
        top_id = candidate_id or 'Unknown'; tactic = ''

    retrieved_context = '\n\n'.join(docs[:3]) if docs else 'No techniques retrieved.'

    prompt = f"""[INST] You are a cybersecurity analyst writing a structured incident report. Respond with valid JSON only.
Use ONLY the information provided below. Do not add external knowledge, CVE references, or technique IDs not present in the retrieved context.
Based on the retrieved ATT&CK techniques and observed network behaviors below, write a structured incident report.

OBSERVED BEHAVIORS:
{(alert_context or '')[:300]}

EXTRACTED RELATIONSHIPS:
{(raw_triples or '')[:300]}

RETRIEVED ATT&CK CONTEXT:
{retrieved_context[:600]}

TOP CANDIDATE TECHNIQUE: {top_id} | {tactic}

Return a JSON report with fields: technique_id, tactic, summary, evidence, next_step.
Use {top_id} as the technique_id unless the retrieved context strongly suggests otherwise. [/INST]
{{"""

    try:
        out = llm(prompt, max_tokens=400, temperature=0.0,
                  seed=RANDOM_SEED, grammar=report_grammar,
                  repeat_penalty=1.0, stop=['[/INST]'])
        raw = '{' + out['choices'][0]['text']
        match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
        raw = match.group(0) if match else raw
        parsed = json.loads(raw)
        return {
            'technique_id': str(parsed.get('technique_id', top_id)),
            'tactic':       str(parsed.get('tactic', tactic)),
            'summary':      str(parsed.get('summary', ''))[:400],
            'evidence':     str(parsed.get('evidence', ''))[:400],
            'next_step':    str(parsed.get('next_step', ''))[:200],
        }
    except Exception as e:
        print(f'  [RAG report] generation failed: {e}')
        return {
            'technique_id': top_id,
            'tactic':       tactic,
            'summary':      (hyde_query or '')[:400],
            'evidence':     retrieved_context[:400],
            'next_step':    'Validate retrieval with telemetry and investigate correlated hosts.',
        }


In [38]:
PURITY_MIN = 0.70

src = community_df if 'community_df' in globals() else df_filtered

community_gt = {}
excluded = {}
for cid in sorted(src[src['community_id'] != -1]['community_id'].unique()):
    g = src[src['community_id'] == cid]
    counts = g[' Label'].value_counts()
    dom_label = counts.index[0]
    dom_share = counts.iloc[0] / len(g)

    if dom_label == 'BENIGN':
        excluded[cid] = f'benign-dominant ({dom_share:.2f})'
        continue
    if dom_share < PURITY_MIN:
        excluded[cid] = f'impure ({dom_share:.2f} < {PURITY_MIN})'
        continue
    gt = label_to_technique.get(dom_label)
    if not gt:
        excluded[cid] = f'no technique for label {dom_label}'
        continue
    community_gt[cid] = gt

eval_cids = sorted(community_gt.keys())

print(f'Scoreable communities: {len(eval_cids)}')
for cid in eval_cids:
    g = src[src['community_id'] == cid]
    print(f'  cmty {cid:>2} | n={len(g):>5} | {community_gt[cid]:<10} | dom={g[" Label"].value_counts().index[0]}')

print(f'\nExcluded: {len(excluded)}')
for cid, reason in excluded.items():
    print(f'  cmty {cid:>2} | {reason}')

print('\nClean ground-truth distribution:')
print(pd.Series(list(community_gt.values())).value_counts().to_string())

Scoreable communities: 9
  cmty  2 | n=  929 | T1110.001  | dom=SSH - Patator
  cmty  5 | n=  234 | T1110.001  | dom=SSH - Patator
  cmty  6 | n=   76 | T1110.001  | dom=SSH - Patator
  cmty 10 | n=    8 | T1046      | dom=PortScan
  cmty 11 | n=    8 | T1046      | dom=PortScan
  cmty 14 | n=    5 | T1046      | dom=PortScan
  cmty 16 | n=    4 | T1046      | dom=PortScan
  cmty 17 | n=    4 | T1046      | dom=PortScan
  cmty 20 | n=    2 | T1046      | dom=PortScan

Excluded: 12
  cmty  0 | impure (0.30 < 0.7)
  cmty  1 | impure (0.50 < 0.7)
  cmty  3 | benign-dominant (1.00)
  cmty  4 | benign-dominant (1.00)
  cmty  7 | benign-dominant (1.00)
  cmty  8 | impure (0.52 < 0.7)
  cmty  9 | impure (0.46 < 0.7)
  cmty 12 | impure (0.67 < 0.7)
  cmty 13 | benign-dominant (0.50)
  cmty 15 | benign-dominant (0.75)
  cmty 18 | benign-dominant (1.00)
  cmty 19 | impure (0.67 < 0.7)

Clean ground-truth distribution:
T1046        6
T1110.001    3


In [39]:
def parent_match(gt_id: str, gen_id: str) -> bool:
    if not gt_id or not gen_id or gen_id == 'Unknown':
        return False
    return gt_id.split('.')[0] == gen_id.split('.')[0]


print(f'Evaluating {len(eval_cids)} scoreable communities\n')

results = []

import numpy as np
from sentence_transformers import util as sutil

if 'tech_ids' not in globals() or 'tech_embs' not in globals():
    tech_ids = []
    tech_texts = []
    for tid, info in attck_techniques.items():
        tactic = (info.get('tactics') or [''])[0] if isinstance(info.get('tactics'), list) else (info.get('tactics') or '')
        text = f"ID: {tid}\nName: {info.get('name','')}\nTactic: {tactic}\nDescription: {info.get('description','')}"
        tech_ids.append(tid)
        tech_texts.append(text)
    tech_embs = embedder.encode(tech_texts, convert_to_tensor=True)

tid_to_idx = {tid: i for i, tid in enumerate(tech_ids)}


def kg_candidate_tids(cid):
    """Candidate techniques come from the community's DOMINANT extracted relation,
    mapped through RELATION_TO_TECHNIQUES. Using only the most frequent relation
    (rather than the union of all relations) avoids candidate-set pollution when
    the LLM hallucinates a multi-stage narrative from repetitive traffic.
    Returns an ordered, de-duplicated list (empty if the community has no triples)."""
    rels = [t.get('relation', '') for t in community_triples.get(str(cid), [])]
    if not rels:
        return []
    dom_rel = Counter(rels).most_common(1)[0][0]
    cands, seen = [], set()
    for tid in RELATION_TO_TECHNIQUES.get(dom_rel, []):
        if tid in tid_to_idx and tid not in seen:
            cands.append(tid); seen.add(tid)
    return cands


def scoped_rerank(hyde_query, raw_triples, alert_context, cid):
    """KG-anchored rerank. Candidate set = techniques linked via the community's
    dominant extracted relation; ranked by similarity to the community's own
    alert text (primary signal) plus the raw triples, with HyDE as a weak tie-break.
    Falls back to the full set only when no triples exist.
    Returns (best_id, best_score, grounded)."""
    cand_tids = kg_candidate_tids(cid)
    grounded  = bool(cand_tids)
    cand_idx  = [tid_to_idx[t] for t in cand_tids] if grounded else list(range(len(tech_ids)))

    cand_embs = tech_embs[cand_idx]
    q_alert = embedder.encode((alert_context or '')[:400], convert_to_tensor=True)
    q_raw   = embedder.encode(raw_triples or '',           convert_to_tensor=True)
    sims_a  = sutil.pytorch_cos_sim(q_alert, cand_embs)[0].cpu().tolist()
    sims_r  = sutil.pytorch_cos_sim(q_raw,   cand_embs)[0].cpu().tolist()
    combined = [max(a, r) for a, r in zip(sims_a, sims_r)]
    local_best = int(np.argmax(combined))
    return tech_ids[cand_idx[local_best]], float(combined[local_best]), grounded


for cid in eval_cids:
    group        = community_df[community_df['community_id'] == cid]
    ground_truth = community_gt[cid]

    hyde_query, raw_triples, alert_context, tactic_scope = build_hyde_query(cid)

    # Dominant relation as an extra short query — gives ChromaDB an explicit
    # behavioural signal alongside the HyDE text and raw triples.
    dom_rel = Counter(t.get('relation', '') for t in community_triples.get(str(cid), [])).most_common(1)
    relation_query = dom_rel[0][0].replace('_', ' ').lower() if dom_rel else ''

    try:
        retrieval = collection.query(
            query_texts=[hyde_query, raw_triples, relation_query],
            n_results=TOP_K
        )
        docs_lists  = retrieval.get('documents', [])
        metas_lists = retrieval.get('metadatas', [])
        docs = []
        metas = []
        for dl in docs_lists:
            docs.extend(dl or [])
        for ml in metas_lists:
            metas.extend(ml or [])
        seen = set(); unique_metas = []
        for m in metas:
            tid = m.get('technique_id') if isinstance(m, dict) else None
            if tid and tid not in seen:
                unique_metas.append(m); seen.add(tid)
        metas = unique_metas
        retrieved_ids = [m.get('technique_id', '') for m in metas]
    except Exception as e:
        print('Collection query failed:', e)
        docs, metas, retrieved_ids = [], [], []

    # KG-anchored rerank: candidate techniques come from the graph's extracted
    # relations; semantic similarity ranks within that set.
    try:
        best_id, best_score, kg_grounded = scoped_rerank(hyde_query, raw_triples, alert_context, cid)
    except Exception as e:
        best_id = 'Unknown'; best_score = 0.0; kg_grounded = False

    rag_report = generate_rag_report(hyde_query, raw_triples, alert_context, docs, metas, best_id)
    rag_id     = rag_report.get('technique_id', best_id)
    rag_tactic = rag_report.get('tactic', '')

    base_report = generate_baseline_report(raw_triples, alert_context, int(cid))
    base_id     = base_report.get('technique_id', 'Unknown')

    retrieval_hit = ground_truth in retrieved_ids
    rag_grounded  = rag_id != 'Unknown'
    rag_exact     = rag_id == ground_truth
    rag_parent    = parent_match(ground_truth, rag_id)
    base_exact    = base_id == ground_truth
    base_parent   = parent_match(ground_truth, base_id)

    try:
        summary_text   = rag_report.get('summary', '') or ''
        retrieved_text = ' '.join(docs[:3]) if docs else ''
        if summary_text and retrieved_text:
            emb_summary   = embedder.encode(summary_text,   convert_to_tensor=True)
            emb_retrieved = embedder.encode(retrieved_text, convert_to_tensor=True)
            faithfulness  = float(sutil.pytorch_cos_sim(emb_summary, emb_retrieved).item())
        else:
            faithfulness = 0.0
    except Exception:
        faithfulness = 0.0

    results.append({
        'community_id':      cid,
        'ground_truth':      ground_truth,
        'dominant_label':    group[' Label'].mode().iloc[0],
        'dominant_tactic':   group['Tactics'].mode().iloc[0],
        'tactic_scope':      tactic_scope or 'none',
        'hyde_query':        hyde_query,
        'raw_triples':       raw_triples,
        'retrieved_ids':     retrieved_ids,
        'retrieval_hit':     retrieval_hit,
        'rag_technique_id':  rag_id,
        'rag_grounded':      rag_grounded,
        'rag_exact_match':   rag_exact,
        'rag_parent_match':  rag_parent,
        'rag_tactic':        rag_report.get('tactic',    ''),
        'rag_summary':       rag_report.get('summary',   ''),
        'rag_evidence':      rag_report.get('evidence',  ''),
        'rag_next_step':     rag_report.get('next_step', ''),
        'rag_score':         best_score,
        'kg_grounded':       kg_grounded,
        'faithfulness':      faithfulness,
        'base_technique_id': base_id,
        'base_exact_match':  base_exact,
        'base_parent_match': base_parent,
        'base_tactic':       base_report.get('tactic',  ''),
        'base_summary':      base_report.get('summary', ''),
    })

    print(f"  Community {cid} [{ground_truth}] scope={tactic_scope} | ")
    print(f"    RAG: {rag_id} (score={best_score:.3f}) | exact={rag_exact} | parent={rag_parent} | faith={faithfulness:.3f}")
    print(f"    Baseline: {base_id} | exact={base_exact} | parent={base_parent}")

print(f'\nDone: {len(results)} communities evaluated')


Evaluating 9 scoreable communities

  [Baseline raw]: '{ "technique_id": "T1046", "tactic": "Discovery", "summary": "The network behavior described involves an attacker brute-forcing credentials on an SSH service, which is indicative of a credential guess'
  Community 2 [T1110.001] scope=credential-access | 
    RAG: T1110.003 (score=0.600) | exact=False | parent=True | faith=0.654
    Baseline: T1046 | exact=False | parent=False
  [Baseline raw]: '{ "technique_id": "T1046", "tactic": "Discovery", "summary": "The network flow targeting SSH on port 22 with minimal control exchange and high packet rate suggests a SYN flood or scan pattern, which c'
  [Baseline Community 5] parse failed: Unterminated string starting at: line 1 column 282 (char 281) | raw: '{ "technique_id": "T1046", "tactic": "Discovery", "summary": "The network flow targeting SSH on port 22 with minimal control exchange and high packet '
  Community 5 [T1110.001] scope=credential-access | 
    RAG: T1110.003 (score=0.587

In [40]:
print("Error attribution: where does each miss originate?\n")

reachable, retr_correct, extract_fail, no_triples = [], [], [], []
for r in results:
    cid = r['community_id']
    gt = r['ground_truth']
    rels = [t.get('relation','') for t in community_triples.get(str(cid), [])]
    if not rels:
        no_triples.append(cid); continue
    dom_rel = Counter(rels).most_common(1)[0][0]
    cand = RELATION_TO_TECHNIQUES.get(dom_rel, [])
    if gt in cand:
        reachable.append(cid)
        if r['rag_exact_match']:
            retr_correct.append(cid)
    else:
        extract_fail.append(cid)

n = len(results)
print(f"Reachable from dominant relation (retrieval CAN succeed): {len(reachable)}/{n}  {reachable}")
print(f"  -> retrieval picked correct technique:                  {len(retr_correct)}/{len(reachable)}  {retr_correct}")
print(f"Extraction failure (correct technique not in candidates): {len(extract_fail)}/{n}  {extract_fail}")
print(f"No triples extracted at all:                              {len(no_triples)}/{n}  {no_triples}")

print(f"\nRetrieval accuracy ON REACHABLE communities (true Stage-5 performance): "
      f"{len(retr_correct)}/{len(reachable)} = {len(retr_correct)/max(len(reachable),1):.1%}")
print(f"Ceiling imposed by extraction (Stage 3): {(len(extract_fail)+len(no_triples))}/{n} "
      f"communities can never be correct regardless of retrieval")

Error attribution: where does each miss originate?

Reachable from dominant relation (retrieval CAN succeed): 7/9  [np.int64(2), np.int64(5), np.int64(6), np.int64(10), np.int64(11), np.int64(14), np.int64(17)]
  -> retrieval picked correct technique:                  2/7  [np.int64(10), np.int64(14)]
Extraction failure (correct technique not in candidates): 1/9  [np.int64(16)]
No triples extracted at all:                              1/9  [np.int64(20)]

Retrieval accuracy ON REACHABLE communities (true Stage-5 performance): 2/7 = 28.6%
Ceiling imposed by extraction (Stage 3): 2/9 communities can never be correct regardless of retrieval


In [41]:
src = community_df if 'community_df' in globals() else df_filtered

rows = []
for cid in sorted(src[src['community_id'] != -1]['community_id'].unique()):
    g = src[src['community_id'] == cid]
    label_counts = g[' Label'].value_counts()
    dom_label = label_counts.index[0]
    dom_share = label_counts.iloc[0] / len(g)
    tech_mode = g['Technique'].mode()
    gt_from_tech = tech_mode.iloc[0] if len(tech_mode) else None
    gt_from_label = label_to_technique.get(dom_label)
    rows.append({
        'cid': cid,
        'size': len(g),
        'dom_label': dom_label,
        'dom_share': round(dom_share, 3),
        'gt_via_Technique_col': gt_from_tech,
        'gt_via_label_map': gt_from_label,
        'mismatch': str(gt_from_tech) != str(gt_from_label),
        'n_labels_in_cluster': g[' Label'].nunique(),
    })

audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
print(f"\nCommunities where Technique col disagrees with dominant-label mapping: "
      f"{audit['mismatch'].sum()} / {len(audit)}")
print(f"Communities that are >50% benign: "
      f"{(audit['dom_label'] == 'BENIGN').sum()}")
print("\nGround-truth distribution actually used in eval (via Technique col):")
print(audit['gt_via_Technique_col'].value_counts().to_string())
print("\nGround-truth distribution if derived from dominant label:")
print(audit['gt_via_label_map'].value_counts(dropna=False).to_string())

 cid  size     dom_label  dom_share gt_via_Technique_col gt_via_label_map  mismatch  n_labels_in_cluster
   0  6548 FTP - Patator      0.300            T1110.001        T1110.001     False                    6
   1  2630          DDoS      0.500            T1498.001        T1498.001     False                    4
   2   929 SSH - Patator      0.997            T1110.001        T1110.001     False                    3
   3   772        BENIGN      0.997                T1046              NaN      True                    2
   4   405        BENIGN      0.998                T1046              NaN      True                    2
   5   234 SSH - Patator      1.000            T1110.001        T1110.001     False                    1
   6    76 SSH - Patator      0.974            T1110.001        T1110.001     False                    2
   7    67        BENIGN      1.000                  NaN              NaN     False                    1
   8    66           Bot      0.515            T1071.00

In [42]:
n = len(results)

# Per-class (per-technique) macro averaging removes the majority-class advantage
# that lets a constant predictor score highly on an imbalanced eval set.
from collections import defaultdict

by_gt_rag = defaultdict(list)
by_gt_base = defaultdict(list)
for r in results:
    by_gt_rag[r['ground_truth']].append(r['rag_exact_match'])
    by_gt_base[r['ground_truth']].append(r['base_exact_match'])

rag_macro  = round(float(np.mean([np.mean(v) for v in by_gt_rag.values()])), 4)
base_macro = round(float(np.mean([np.mean(v) for v in by_gt_base.values()])), 4)

# Majority-class baseline: always predict the most frequent ground-truth label.
gt_counts       = Counter(r['ground_truth'] for r in results)
majority_label  = gt_counts.most_common(1)[0][0]
majority_acc    = round(gt_counts[majority_label] / n, 4)

metrics = {
    'model':                       'qwen2.5-3b-instruct-q4_k_m.gguf',
    'retrieval_strategy':          'HyDE + KG-anchored candidate set (relation bridge)',
    'embed_model':                 EMBED_MODEL,
    'n_communities_evaluated':     n,
    'top_k_retrieval':             TOP_K,
    'retrieval_hit_rate':          round(sum(r['retrieval_hit']    for r in results) / n, 4),
    'rag_grounding_rate':          round(sum(r['rag_grounded']     for r in results) / n, 4),
    'rag_exact_match_rate':        round(sum(r['rag_exact_match']  for r in results) / n, 4),
    'rag_parent_match_rate':       round(sum(r['rag_parent_match'] for r in results) / n, 4),
    'rag_exact_macro':             rag_macro,
    'rag_faithfulness_mean':       round(sum(r.get('faithfulness', 0) for r in results) / n, 4),
    'baseline_exact_match_rate':   round(sum(r['base_exact_match'] for r in results) / n, 4),
    'baseline_parent_match_rate':  round(sum(r['base_parent_match']for r in results) / n, 4),
    'baseline_exact_macro':        base_macro,
    'majority_class_label':        majority_label,
    'majority_class_accuracy':     majority_acc,
}
metrics['delta_exact_match']  = round(metrics['rag_exact_match_rate']  - metrics['baseline_exact_match_rate'],  4)
metrics['delta_parent_match'] = round(metrics['rag_parent_match_rate'] - metrics['baseline_parent_match_rate'], 4)
metrics['delta_exact_macro']  = round(metrics['rag_exact_macro']       - metrics['baseline_exact_macro'],       4)

grounded = [r for r in results if r.get('kg_grounded')]
if grounded:
    g_exact = round(sum(r['rag_exact_match'] for r in grounded) / len(grounded), 4)
    g_parent = round(sum(r['rag_parent_match'] for r in grounded) / len(grounded), 4)
    by_gt_g = defaultdict(list)
    for r in grounded:
        by_gt_g[r['ground_truth']].append(r['rag_exact_match'])
    g_macro = round(float(np.mean([np.mean(v) for v in by_gt_g.values()])), 4)
else:
    g_exact = g_parent = g_macro = 0.0
metrics['n_kg_grounded']             = len(grounded)
metrics['rag_grounded_exact_match']  = g_exact
metrics['rag_grounded_parent_match'] = g_parent
metrics['rag_grounded_exact_macro']  = g_macro

pd.DataFrame(results).to_csv(RESULTS_DIR / 'rag_reports.csv', index=False)
with open(RESULTS_DIR / 'rag_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('METRICS')
print(json.dumps(metrics, indent=2))
print(f"\nMicro (raw) accuracy — sensitive to class imbalance:")
print(f"  RAG exact:        {metrics['rag_exact_match_rate']:.2%}")
print(f"  Baseline exact:   {metrics['baseline_exact_match_rate']:.2%}")
print(f"  Majority class:   {metrics['majority_class_accuracy']:.2%}  (always predict {majority_label})")
print(f"\nMacro accuracy — equal weight per technique, removes majority bias:")
print(f"  RAG macro:        {metrics['rag_exact_macro']:.2%}")
print(f"  Baseline macro:   {metrics['baseline_exact_macro']:.2%}")
print(f"  Delta (macro):    {metrics['delta_exact_macro']:+.2%}")
print(f"\nParent-technique match:")
print(f"  RAG parent:       {metrics['rag_parent_match_rate']:.2%}")
print(f"  Baseline parent:  {metrics['baseline_parent_match_rate']:.2%}")
print(f"\nRAG faithfulness:   {metrics['rag_faithfulness_mean']:.4f}")
print(f"Retrieval hit:      {metrics['retrieval_hit_rate']:.2%}")
print(f"\nKG-grounded only ({metrics['n_kg_grounded']}/{n} communities had triples):")
print(f"  Grounded exact:   {metrics['rag_grounded_exact_match']:.2%}")
print(f"  Grounded parent:  {metrics['rag_grounded_parent_match']:.2%}")
print(f"  Grounded macro:   {metrics['rag_grounded_exact_macro']:.2%}")


METRICS
{
  "model": "qwen2.5-3b-instruct-q4_k_m.gguf",
  "retrieval_strategy": "HyDE + KG-anchored candidate set (relation bridge)",
  "embed_model": "all-MiniLM-L6-v2",
  "n_communities_evaluated": 9,
  "top_k_retrieval": 10,
  "retrieval_hit_rate": 0.4444,
  "rag_grounding_rate": 1.0,
  "rag_exact_match_rate": 0.2222,
  "rag_parent_match_rate": 0.5556,
  "rag_exact_macro": 0.1667,
  "rag_faithfulness_mean": 0.5779,
  "baseline_exact_match_rate": 0.5556,
  "baseline_parent_match_rate": 0.5556,
  "baseline_exact_macro": 0.4167,
  "majority_class_label": "T1046",
  "majority_class_accuracy": 0.6667,
  "delta_exact_match": -0.3334,
  "delta_parent_match": 0.0,
  "delta_exact_macro": -0.25,
  "n_kg_grounded": 8,
  "rag_grounded_exact_match": 0.25,
  "rag_grounded_parent_match": 0.625,
  "rag_grounded_exact_macro": 0.2
}

Micro (raw) accuracy — sensitive to class imbalance:
  RAG exact:        22.22%
  Baseline exact:   55.56%
  Majority class:   66.67%  (always predict T1046)

Macro accu